# Project framing and offline contract

This notebook teaches the complete offline research workflow for one-day-ahead lower-tail Value-at-Risk (VaR) and Expected Shortfall (ES).

Define symbols before use:

- $R_{t+1}$: next-day log return.
- $lpha$: left-tail probability, for example $lpha = 0.05$.
- $q_{lpha,t+1}$: forecast $lpha$-quantile of next-day return.
- $\mathrm{VaR}_{lpha,t+1} = -q_{lpha,t+1}$ on the public positive-loss scale.

The offline contract is strict:

1. ClickHouse is **not required** for notebook execution.
2. Tracked raw parquet files in `data/raw/` are the only required input.
3. Database fetching is an optional one-time cache-refresh path documented separately.


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from conformal_var_risk.config import load_config
from conformal_var_risk.data.raw_cache import load_raw_minute_bars
from conformal_var_risk.data.daily_panel import build_daily_log_return_panel, _filter_regular_session
from conformal_var_risk.data.realized_variance import build_daily_realized_variance_panel
from conformal_var_risk.features.feature_builder import build_feature_table
from conformal_var_risk.models.registry import build_model_factories
from conformal_var_risk.evaluation.backtest import run_backtest
from conformal_var_risk.evaluation.metrics import compute_summary_metrics

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config.toml"
config = load_config(CONFIG_PATH)

print(f"project_root={PROJECT_ROOT}")
print(f"raw_dir={config.raw_data.raw_dir}")
print(f"symbols={config.portfolio.symbols}")
print(f"date_range=[{config.run.start_date}, {config.run.end_date}]")

project_root=/home/ai4000/projects/one-time-projects/conformal-var-risk
raw_dir=data/raw
symbols=['AAPL', 'JPM', 'TSLA', 'SPY']
date_range=[2019-01-01, 2023-12-31]


# Raw parquet inspection

The raw cache contract requires columns `symbol`, `ts`, and `close`.

Define notation used in later steps:

- `symbol`: asset identifier.
- `ts`: timestamp in Coordinated Universal Time (UTC).
- `close`: minute close price in level units.

The next code cell loads raw rows from local parquet files only and checks the sample bounds.


In [2]:
raw_minute_bars = load_raw_minute_bars(
    raw_dir=PROJECT_ROOT / config.raw_data.raw_dir,
    file_pattern=config.raw_data.file_pattern,
    required_symbols=config.portfolio.symbols,
    start_date=config.run.start_date,
    end_date=config.run.end_date,
)

print(raw_minute_bars.head(3))
print(raw_minute_bars.tail(3))
print(f"rows={len(raw_minute_bars):,}")
print(f"symbols={sorted(raw_minute_bars['symbol'].unique())}")
print(f"first_ts={raw_minute_bars['ts'].min()}")
print(f"last_ts={raw_minute_bars['ts'].max()}")

  symbol                        ts       close
0   AAPL 2019-01-01 14:30:00+00:00  150.019110
1   AAPL 2019-01-01 14:31:00+00:00  150.007317
2   AAPL 2019-01-01 14:32:00+00:00  150.001011
        symbol                        ts       close
2034237   TSLA 2023-12-29 20:57:00+00:00  175.155016
2034238   TSLA 2023-12-29 20:58:00+00:00  175.127912
2034239   TSLA 2023-12-29 20:59:00+00:00  175.125925
rows=2,034,240
symbols=['AAPL', 'JPM', 'SPY', 'TSLA']
first_ts=2019-01-01 14:30:00+00:00
last_ts=2023-12-29 20:59:00+00:00


# Regular-session filtering and daily close construction

Raw minute data can include pre-market or after-hours observations, so we filter to the regular United States equity session and then keep the last close per symbol per day.

Define symbols:

- $P_{t,i}$: minute close level for day $t$ and minute index $i$.
- $P^{\mathrm{close}}_t$: last regular-session close on day $t$.

The next code cell shows this transformation explicitly before any return formula is applied.


In [3]:
regular_session = _filter_regular_session(raw_minute_bars)
working = regular_session.copy()
working["date"] = (
    working["ts"].dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
)

daily_close = (
    working.sort_values(["symbol", "ts"])
    .groupby(["date", "symbol"], as_index=False)["close"]
    .last()
    .pivot(index="date", columns="symbol", values="close")
    .sort_index()
)

print(daily_close.head())
print(daily_close.tail())
print(f"daily_close_shape={daily_close.shape}")

symbol            AAPL         JPM         SPY       TSLA
date                                                     
2019-01-01  152.063355  108.482020  274.655830  58.958926
2019-01-02  152.201802  109.267069  277.125378  57.260835
2019-01-03  153.228634  108.603288  278.468280  60.761836
2019-01-04  154.866874  107.953780  279.139779  61.371968
2019-01-07  154.311995  107.247008  274.305765  61.639348
symbol            AAPL        JPM         SPY        TSLA
date                                                     
2023-12-25  989.170648  89.346278  615.242301  164.803708
2023-12-26  992.276259  88.734337  615.536069  169.809197
2023-12-27  978.971095  88.532340  623.129816  170.312816
2023-12-28  985.857374  88.287669  630.224161  178.068521
2023-12-29  985.975237  85.994024  620.041229  175.125925
daily_close_shape=(1304, 4)


# Daily log-return construction

Define daily log return for symbol $s$:

$$
r_{t,s} = \log\left(rac{P^{\mathrm{close}}_{t,s}}{P^{\mathrm{close}}_{t-1,s}}ight).
$$

Log returns are additive across time and standard in quantitative finance research pipelines.

The next code cell builds the full daily return panel and the equal-weight `portfolio` series.


In [4]:
daily_returns = build_daily_log_return_panel(
    minute_bars=raw_minute_bars,
    symbols=config.portfolio.symbols,
)

print(daily_returns.head())
print(daily_returns.tail())
print(f"daily_returns_shape={daily_returns.shape}")

symbol          AAPL       JPM      TSLA       SPY  portfolio
date                                                         
2019-01-02  0.000910  0.007211 -0.029224  0.008951  -0.003038
2019-01-03  0.006724 -0.006093  0.059345  0.004834   0.016202
2019-01-04  0.010635 -0.005999  0.009991  0.002409   0.004259
2019-01-07 -0.003589 -0.006569  0.004347 -0.017469  -0.005820
2019-01-08  0.000088  0.008191  0.052646  0.022829   0.020938
symbol          AAPL       JPM      TSLA       SPY  portfolio
date                                                         
2023-12-25  0.002352 -0.001502 -0.015419 -0.008356  -0.005731
2023-12-26  0.003135 -0.006873  0.029920  0.000477   0.006665
2023-12-27 -0.013499 -0.002279  0.002961  0.012261  -0.000139
2023-12-28  0.007010 -0.002767  0.044532  0.011321   0.015024
2023-12-29  0.000120 -0.026323 -0.016663 -0.016290  -0.014789
daily_returns_shape=(1303, 5)


# Realized variance construction

Define intraday log return and daily realized variance:

- $r_{t,i}$: intraday log return for interval $i$ on day $t$.
- $\mathrm{RV}_t$: daily realized variance.

Formula:

$$
\mathrm{RV}_t = \sum_i r_{t,i}^2.
$$

For the equal-weight portfolio, the pipeline first averages synchronized constituent intraday returns and then sums their squares. This keeps cross-asset covariance terms; averaging constituent variances would not. Values are daily variance in squared decimal-return units and are not annualized.

The next code cell computes one realized-variance series per asset and the covariance-aware portfolio series.


In [5]:
daily_realized_variance = build_daily_realized_variance_panel(
    minute_bars=raw_minute_bars,
    symbols=config.portfolio.symbols,
)

print(daily_realized_variance.head())
print(daily_realized_variance.tail())
print(f"daily_realized_variance_shape={daily_realized_variance.shape}")

symbol          AAPL       JPM      TSLA       SPY     portfolio
date                                                            
2019-01-01  0.000003  0.000002  0.000008  0.000002  1.079647e-06
2019-01-02  0.000002  0.000002  0.000010  0.000001  7.834164e-07
2019-01-03  0.000002  0.000002  0.000017  0.000001  1.373361e-06
2019-01-04  0.000003  0.000002  0.000008  0.000001  8.640528e-07
2019-01-07  0.000002  0.000002  0.000008  0.000002  9.083480e-07
symbol          AAPL       JPM      TSLA       SPY     portfolio
date                                                            
2023-12-25  0.000002  0.000001  0.000008  0.000001  8.763950e-07
2023-12-26  0.000002  0.000002  0.000010  0.000001  8.432534e-07
2023-12-27  0.000003  0.000001  0.000008  0.000001  8.453520e-07
2023-12-28  0.000002  0.000001  0.000013  0.000001  1.400022e-06
2023-12-29  0.000002  0.000003  0.000008  0.000002  1.336145e-06
daily_realized_variance_shape=(1304, 5)


# Feature building

We keep a small feature set and enforce no-lookahead timing.

For forecasting day $t+1$, every feature at row $t$ uses information available through day $t$ only.

Feature examples in this project:

- lagged realized variance
- 5-day mean realized variance
- 21-day mean realized variance
- lagged absolute return
- optional 5-day realized return volatility

The next code cell materializes the long-form feature table.


In [6]:
aligned_rv = daily_realized_variance.reindex(daily_returns.index)

feature_table = build_feature_table(
    daily_returns=daily_returns,
    realized_variance=aligned_rv,
    include_realized_return_volatility_5d=config.features.include_realized_return_volatility_5d,
)

print(feature_table.head())
print(feature_table.tail())
print(f"feature_table_shape={feature_table.shape}")
print(f"feature_columns={list(feature_table.columns)}")

        date asset  target_next_day_return      rv_t  rv_lag_1  rv_mean_5  \
0 2019-01-31  AAPL                0.000775  0.000002  0.000002   0.000002   
1 2019-02-01  AAPL                0.019873  0.000002  0.000002   0.000002   
2 2019-02-04  AAPL               -0.005662  0.000003  0.000002   0.000002   
3 2019-02-05  AAPL                0.005548  0.000002  0.000003   0.000003   
4 2019-02-06  AAPL               -0.001368  0.000002  0.000002   0.000002   

   rv_mean_21  abs_return_lag_1  return_volatility_5  
0    0.000003          0.001118             0.003647  
1    0.000003          0.002695             0.003937  
2    0.000003          0.000775             0.003067  
3    0.000003          0.019873             0.010434  
4    0.000003          0.005662             0.010138  
           date      asset  target_next_day_return          rv_t  \
6400 2023-12-22  portfolio               -0.005731  8.320622e-07   
6401 2023-12-25  portfolio                0.006665  8.763950e-07   
640

# Training and backtest setup

The backtest is one-step-ahead and walk-forward.

Define symbols:

- $W$: calibration window length.
- $lpha$: VaR tail level.
- $\hat{q}_{lpha,t+1}$: forecast lower quantile for next day.

At each evaluation date, models fit on the trailing window, forecast next-day lower tail, and then receive the realized return.


In [7]:
model_factories = build_model_factories(config=config)

print(f"model_names={list(model_factories.keys())}")
print(f"calibration_window={config.backtest.calibration_window}")
print(f"alphas={config.backtest.alphas}")

usable_rows = len(daily_returns)
expected_evaluation_rows = usable_rows - config.backtest.calibration_window
print(f"usable_daily_rows={usable_rows}")
print(f"expected_rows_per_asset_model_alpha={expected_evaluation_rows}")

model_names=['historical', 'garch_normal', 'garch_student_t', 'filtered_historical', 'conformal']
calibration_window=1150
alphas=[0.05, 0.01]
usable_daily_rows=1303
expected_rows_per_asset_model_alpha=153


# Model execution

This is the model run itself. We execute all configured models and tail levels on the same daily panel.

The output is a long table with one row per date, asset, model, and tail level, including:

- predicted lower quantile
- predicted VaR and ES
- realized return
- violation indicator


In [8]:
backtest_results = run_backtest(
    returns_by_asset=daily_returns,
    alphas=config.backtest.alphas,
    calibration_window=config.backtest.calibration_window,
    model_factories=model_factories,
)

print(backtest_results.head())
print(backtest_results.tail())
print(f"backtest_results_shape={backtest_results.shape}")

        date asset       model  alpha  predicted_var  predicted_es  \
0 2023-05-31  AAPL  historical   0.05       0.022702      0.030396   
1 2023-05-31  AAPL  historical   0.01       0.036017      0.037919   
2 2023-06-01  AAPL  historical   0.05       0.022702      0.030396   
3 2023-06-01  AAPL  historical   0.01       0.036017      0.037919   
4 2023-06-02  AAPL  historical   0.05       0.022702      0.030396   

   predicted_lower_quantile  interval_lower  interval_upper  actual_return  \
0                 -0.022702       -0.022702        0.025131       0.007051   
1                 -0.036017       -0.036017        0.035475       0.007051   
2                 -0.022702       -0.022702        0.025131      -0.002270   
3                 -0.036017       -0.036017        0.035475      -0.002270   
4                 -0.022702       -0.022702        0.025131      -0.006424   

   violation  
0      False  
1      False  
2      False  
3      False  
4      False  
           date     

# Evaluation metrics

We evaluate calibration and tail-loss quality by period.

Core metrics include:

- violation rate, coverage rate, and an exact 95% Clopper-Pearson interval
- lower-tail quantile loss
- Christoffersen coverage tests
- ES diagnostics

The next code cell computes period-level summary metrics from the backtest table.


In [9]:
summary_metrics = compute_summary_metrics(
    backtest_results=backtest_results,
    evaluation_config=config.evaluation,
)

print(summary_metrics.head())
print(summary_metrics.tail())
print(f"summary_metrics_shape={summary_metrics.shape}")

  period asset                model  alpha  observations  violations  \
0   full  AAPL            conformal   0.01           153           0   
1   full  AAPL            conformal   0.05           153           4   
2   full  AAPL  filtered_historical   0.01           153           1   
3   full  AAPL  filtered_historical   0.05           153           3   
4   full  AAPL         garch_normal   0.01           153           1   

   violation_rate  violation_rate_ci_95_lower  violation_rate_ci_95_upper  \
0        0.000000                    0.000000                    0.023822   
1        0.026144                    0.007168                    0.065584   
2        0.006536                    0.000165                    0.035877   
3        0.019608                    0.004062                    0.056231   
4        0.006536                    0.000165                    0.035877   

   coverage_rate  ...  es_z1_statistic  es_z1_p_value  es_z2_statistic  \
0       1.000000  ...         

# Interpretation and interview narrative

This final section turns metrics into a short interview narrative:

1. Compare coverage and quantile-loss behavior by model.
2. Check whether results are directionally sensible in stress periods.
3. Explain why adaptive conformal updates can improve calibration stability.

The next code cell creates a concise comparison table you can discuss in an interview.


In [10]:
full_period = summary_metrics.loc[summary_metrics["period"] == "full"].copy()
full_period = full_period.sort_values(["asset", "alpha", "avg_quantile_loss"])

comparison_columns = [
    "asset",
    "model",
    "alpha",
    "violation_rate",
    "coverage_rate",
    "avg_quantile_loss",
    "avg_predicted_var",
]

comparison_table = full_period[comparison_columns].reset_index(drop=True)
print(comparison_table.head(20))

best_by_asset_alpha = (
    full_period.groupby(["asset", "alpha"], as_index=False)["avg_quantile_loss"]
    .min()
    .rename(columns={"avg_quantile_loss": "best_avg_quantile_loss"})
)
print("\nBest quantile-loss by asset and alpha:")
print(best_by_asset_alpha)

   asset                model  alpha  violation_rate  coverage_rate  \
0   AAPL           historical   0.01        0.000000       1.000000   
1   AAPL  filtered_historical   0.01        0.006536       0.993464   
2   AAPL         garch_normal   0.01        0.006536       0.993464   
3   AAPL      garch_student_t   0.01        0.006536       0.993464   
4   AAPL            conformal   0.01        0.000000       1.000000   
5   AAPL           historical   0.05        0.045752       0.954248   
6   AAPL  filtered_historical   0.05        0.019608       0.980392   
7   AAPL      garch_student_t   0.05        0.019608       0.980392   
8   AAPL         garch_normal   0.05        0.019608       0.980392   
9   AAPL            conformal   0.05        0.026144       0.973856   
10   JPM      garch_student_t   0.01        0.013072       0.986928   
11   JPM         garch_normal   0.01        0.013072       0.986928   
12   JPM  filtered_historical   0.01        0.000000       1.000000   
13   J